# EX02. [추가연습] 다중분류 실무전용 문제 모음 - 아이리스 품종

> **실무 전용 노트북** - 이론 설명 없이 TODO 문제만 모았습니다. (관련 이론 노트북: EX02_다중분류_실전문제_아이리스품종.ipynb)
이미 개념은 이해했다는 전제로, 손으로 더 많이 풀어보며 완전히 몸에 익히는 것이 목표입니다. 정답을 먼저 보지 말고 반드시 스스로 코드를 작성해본 뒤 펼쳐서 비교하세요.

EX02 이론 노트북과 같은 데이터를 쓰지만, **다른 모델 조합과 딥러닝 구조**로 새로운 문제를 풀어봅니다.

## 목차
1. 모델 학습 연습
2. 딥러닝 연습
3. 모델 비교 시각화

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

iris = pd.read_csv('../data/iris.csv')
X = iris.drop(columns=['target'])
y = iris['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
X_train.shape

## 1. 모델 학습 연습

**문제 1.** `DecisionTreeClassifier(max_depth=4, random_state=42)`를 학습시켜 accuracy를 구하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
dt = DecisionTreeClassifier(max_depth=4, random_state=42)  # max_depth로 깊이를 제한해 과적합을 방지
dt.fit(X_train_s, y_train)
print(accuracy_score(y_test, dt.predict(X_test_s)))
```

</details>

**문제 2.** `XGBClassifier(n_estimators=100, random_state=42)`를 학습시켜 classification_report를 출력하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
xgb = XGBClassifier(n_estimators=100, random_state=42, eval_metric='mlogloss')  # 다중분류이므로 eval_metric='mlogloss'(이진분류의 logloss에 대응)를 지정
xgb.fit(X_train_s, y_train)
print(classification_report(y_test, xgb.predict(X_test_s)))  # average 옵션 없이도 클래스별 precision/recall/f1을 모두 확인 가능
```

</details>

## 2. 딥러닝 연습

**문제 3.** 입력층 8(relu)→출력 3(softmax)인 작은 모델을 만들어 학습시키고 accuracy를 구하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
import tensorflow as tf
tf.random.set_seed(100)  # 딥러닝은 가중치 초기화가 무작위라 결과가 매번 조금씩 달라질 수 있어 시드를 고정
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
model = Sequential()
model.add(Dense(8, activation='relu', input_shape=(X_train_s.shape[1],)))
model.add(Dense(3, activation='softmax'))  # 다중분류 출력층: 노드 수=클래스 수(3), softmax
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
model.fit(X_train_s, y_train, epochs=100, batch_size=8, validation_data=(X_test_s, y_test), callbacks=[es], verbose=0)
pred = np.argmax(model.predict(X_test_s), axis=1)  # softmax 확률 벡터를 argmax로 최종 클래스 라벨로 변환
print(accuracy_score(y_test, pred))
```

</details>

## 3. 모델 비교 시각화

**문제 4.** `DecisionTreeClassifier`의 feature_importances_를 막대그래프로 그리세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
imp = pd.Series(dt.feature_importances_, index=X.columns)
imp.sort_values().plot(kind='barh')  # sort_values()로 정렬해야 막대그래프가 크기순으로 깔끔하게 보임
plt.show()
```

</details>